In [18]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [19]:
! mkdir -p binary_to_bcd
! cd binary_to_bcd && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0   7348      0 --:--:-- --:--:-- --:--:--  7383


In [20]:
# Iteration 1 prompt
verilog_generation_prompt = """
Generate synthesizable Verilog (no delays, no initial blocks, no always_ff).
Return ONLY code: module ... endmodule.

The testbench expects EXACTLY:
module name: binary_to_bcd_converter
ports:
  input  [4:0] binary_input
  output [7:0] bcd_output  // bcd_output[7:4]=tens, [3:0]=ones

Function: for binary_input 0..31, output BCD where:
  bcd_output[3:0] = binary_input % 10
  bcd_output[7:4] = binary_input / 10

Use combinational logic (always @*).
"""


In [ ]:
from openai import OpenAI
import os
client = OpenAI(
    api_key = ""
)
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": verilog_generation_prompt}],
    max_tokens=1024,
)

raw_llm_output = completion.choices[0].message.content

print("=== RAW LLM OUTPUT ===")
print(raw_llm_output)


=== RAW LLM OUTPUT ===
```verilog
module binary_to_bcd_converter(
    input  [4:0] binary_input,
    output [7:0] bcd_output
);

wire [3:0] tens;
wire [3:0] ones;

assign ones = binary_input % 10;
assign tens = binary_input / 10;

assign bcd_output = {tens, ones};

endmodule
```


In [22]:
import re

m = re.search(r"(?s)\bmodule\b.*?\bendmodule\b", raw_llm_output)
if not m:
    raise ValueError("Could not find a Verilog module in the LLM output.")

verilog_code = m.group(0)

print("=== EXTRACTED VERILOG ===")
print(verilog_code)


=== EXTRACTED VERILOG ===
module binary_to_bcd_converter(
    input  [4:0] binary_input,
    output [7:0] bcd_output
);

wire [3:0] tens;
wire [3:0] ones;

assign ones = binary_input % 10;
assign tens = binary_input / 10;

assign bcd_output = {tens, ones};

endmodule


In [23]:
import os

os.makedirs("binary_to_bcd", exist_ok=True)
design_path = "binary_to_bcd/design.v"

with open(design_path, "w") as f:
    f.write(verilog_code)

print("Wrote:", design_path)


Wrote: binary_to_bcd/design.v


In [24]:
!cd binary_to_bcd && iverilog -g2012 -o sim.vvp design.v binary_to_bcd_tb.v && vvp sim.vvp


Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!
